# 04_03 Classification - NaiveBayes
Train and evaluate NaiveBayes.

[COMMAND_SO]
Command 1

[COMMAND_MUC_DICH]
- Muc tieu nghiep vu: Train NaiveBayes va visual ket qua de so sanh voi cac classifier khac.
- Muc tieu ky thuat: Hien thi bang metric va confusion matrix tren test set.

In [1]:
from pathlib import Path
import json
from pyspark.sql import SparkSession
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
spark=(SparkSession.builder.appName('04_03_nb').master('local[2]').config('spark.sql.shuffle.partitions','16').getOrCreate())
spark.sparkContext.setLogLevel('WARN')
PROJECT_ROOT=Path.cwd().resolve().parent if Path.cwd().name=='notebooks' else Path.cwd().resolve()
FEATURE_DIR=PROJECT_ROOT/'data'/'processed'/'features'
MODEL_DIR=PROJECT_ROOT/'models'/'classification'/'naive_bayes'
METRIC_DIR=PROJECT_ROOT/'reports'/'model_metrics'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
train_df=spark.read.parquet(str(FEATURE_DIR/'classification_train')).select('order_id','label','features').dropna()
val_df=spark.read.parquet(str(FEATURE_DIR/'classification_val')).select('order_id','label','features').dropna()
test_df=spark.read.parquet(str(FEATURE_DIR/'classification_test')).select('order_id','label','features').dropna()
nb=NaiveBayes(featuresCol='features',labelCol='label',modelType='multinomial',smoothing=1.0)
m=nb.fit(train_df)
pred_val=m.transform(val_df)
pred_test=m.transform(test_df)
val_f1=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='f1').evaluate(pred_val)
val_acc=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='accuracy').evaluate(pred_val)
test_f1=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='f1').evaluate(pred_test)
test_acc=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='accuracy').evaluate(pred_test)
metrics={'model_family':'classification','model_name':'NaiveBayes','val_f1':float(val_f1),'val_accuracy':float(val_acc),'f1':float(test_f1),'accuracy':float(test_acc),'test_f1':float(test_f1),'test_accuracy':float(test_acc),'train_rows':train_df.count(),'val_rows':val_df.count(),'test_rows':test_df.count()}
print(metrics)
display(pd.DataFrame([metrics]))
cm_pdf=pred_test.groupBy('label','prediction').count().toPandas()
if not cm_pdf.empty:
    cm_table=cm_pdf.pivot(index='label', columns='prediction', values='count').fillna(0).sort_index().sort_index(axis=1)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm_table, annot=True, fmt='.0f', cmap='Blues')
    plt.title('Confusion Matrix - NaiveBayes (Test)')
    plt.xlabel('Prediction')
    plt.ylabel('Label')
    plt.tight_layout()
    plt.show()
m.write().overwrite().save(str(MODEL_DIR))
(METRIC_DIR/'classification_naive_bayes.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/03 00:36:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/04/03 00:36:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/03 00:36:33 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/03 00:36:33 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/03 00:36:33 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


26/04/03 00:36:35 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/03 00:36:35 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


{'model_family': 'classification', 'model_name': 'NaiveBayes', 'val_f1': 0.85453776332606, 'val_accuracy': 0.8667203003486189, 'f1': 0.8605208129645905, 'accuracy': 0.8718825422365245, 'test_f1': 0.8605208129645905, 'test_accuracy': 0.8718825422365245, 'train_rows': 69609, 'val_rows': 14916, 'test_rows': 14916}


,model_family,model_name,val_f1,val_accuracy,f1,accuracy,test_f1,test_accuracy,train_rows,val_rows,test_rows
0,classification,NaiveBayes,0.854538,0.86672,0.860521,0.871883,0.860521,0.871883,69609,14916,14916


/var/folders/07/jh7lmpq57r72h5jdjy3vwy740000gn/T/ipykernel_888/3390346419.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


336